In [ ]:
from models_v3 import MaskedAutoEncoder
from torch.utils.data import DataLoader
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = MaskedAutoEncoder()
model.load_state_dict(torch.load("model_weights/pretrain_checkpoints/model_13_epoch_800.pt"))
model.eval()

In [ ]:
from torchvision.transforms import v2
from medmnist import PathMNIST

tf = v2.Compose([
    v2.ToTensor(),
    #v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_dataset = PathMNIST(root="./data/",split="test",transform=tf,download=True,size=64)

In [ ]:
from torch.utils.data import RandomSampler

test_samples = RandomSampler(test_dataset,num_samples=5)
test_loader = DataLoader(test_dataset, batch_size=1, sampler=test_samples)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
model.to(device)
for sample,_ in test_loader:
    sample = sample.to(device)
    reconstructed_sample,loss_mask,_ = model(sample)
    original_image = np.clip(np.reshape(sample.detach().cpu(),(3,64,64)).permute(1, 2, 0), 0, 1)
    visible_patches = np.clip(np.reshape(sample.detach().cpu()*(-1*(loss_mask.detach().cpu()-torch.ones(loss_mask.shape))),(3,64,64)).permute(1, 2, 0), 0, 1)
    reconstructed_patches = np.clip(np.reshape(reconstructed_sample.detach().cpu()*loss_mask.detach().cpu(),(3,64,64)).permute(1, 2, 0), 0, 1)
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 4))
    axes.flat[0].imshow(original_image)
    axes.flat[0].set_title("Original Image")
    axes.flat[1].imshow(visible_patches)
    axes.flat[1].set_title("Visible Patches")
    axes.flat[2].imshow(reconstructed_patches)
    axes.flat[2].set_title("Reconstructed Patches")
    axes.flat[3].imshow(reconstructed_patches+visible_patches)
    axes.flat[3].set_title("Reconstructed Image")
    stats_title = (f"Test Sample")
    plt.suptitle(stats_title, fontsize=14, fontweight='bold', y=1.06)
    plt.tight_layout()
    plt.show()
    plt.close(fig)